<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/Uncovering_Human_Mobility_from_Pervasive_Sensing_MIT8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Assuming the data is in a CSV file named 'sensing_data.csv'
# Please replace 'sensing_data.csv' with the actual path to your data file.
# If the file format is not CSV, please adjust pd.read_csv to the appropriate function (e.g., pd.read_json, pd.read_excel).
# If there are specific loading parameters (e.g., delimiter, header), add them here.
df = pd.read_csv('sensing_data.csv')

print("Data loaded successfully into a DataFrame. Displaying the first 5 rows:")
df.head()

In [ ]:
import pandas as pd
import numpy as np

# Acknowledging the FileNotFoundError from the previous attempt.
# For demonstration purposes, creating a sample DataFrame.
# User should replace this with their actual data loading code.

# Creating a sample DataFrame for pervasive sensing data
np.random.seed(42)
data_size = 100
timestamps = pd.to_datetime(pd.date_range(start='2023-01-01', periods=data_size, freq='min'))

sample_data = {
    'timestamp': timestamps,
    'device_id': np.random.choice(['device_A', 'device_B', 'device_C'], data_size),
    'latitude': 34.0522 + (np.random.rand(data_size) - 0.5) * 0.1,
    'longitude': -118.2437 + (np.random.rand(data_size) - 0.5) * 0.1,
    'acceleration_x': np.random.normal(0, 0.5, data_size),
    'acceleration_y': np.random.normal(0, 0.5, data_size),
    'acceleration_z': np.random.normal(9.8, 0.5, data_size),
    'wifi_ssid': np.random.choice(['home_wifi', 'work_wifi', 'public_wifi', np.nan], data_size, p=[0.3, 0.3, 0.3, 0.1]),
    'bluetooth_strength': np.random.randint(-90, -30, data_size)
}
df = pd.DataFrame(sample_data)

print("Sample DataFrame created. Please replace this with your actual data loading.")
print("Displaying the first 5 rows of the sample data:")
df.head()

In [ ]:
print("\n--- DataFrame Info ---")
df.info()

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Descriptive Statistics for Numerical Columns ---")
print(df.describe())

print("\n--- Unique Values and Counts for Categorical Columns ---")
for col in df.select_dtypes(include='object').columns:
    print(f"\nColumn '{col}':")
    print(df[col].value_counts(dropna=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select numerical columns for visualization
numerical_cols = ['latitude', 'longitude', 'acceleration_x', 'acceleration_y', 'acceleration_z', 'bluetooth_strength']

# Set up the matplotlib figure for histograms
plt.figure(figsize=(15, 10))
plt.suptitle('Histograms of Numerical Features', y=1.02, fontsize=16)
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.histplot(df[col], kde=True)
    plt.title(col)
plt.tight_layout()
plt.show()

# Set up the matplotlib figure for box plots
plt.figure(figsize=(15, 10))
plt.suptitle('Box Plots of Numerical Features', y=1.02, fontsize=16)
for i, col in enumerate(numerical_cols):
    plt.subplot(2, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(col)
plt.tight_layout()
plt.show()

print("Histograms and Box Plots generated for numerical columns to identify potential outliers.")

In [ ]:
df['wifi_ssid'] = df['wifi_ssid'].fillna('Unknown')

print("Missing values in 'wifi_ssid' column after filling:")
print(df['wifi_ssid'].isnull().sum())

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.set_index('timestamp')
df = df.sort_index()

print("DataFrame info after setting and sorting by timestamp index:")
df.info()
print("First 5 rows after sorting:")
df.head()

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Radius of Earth in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

# Calculate distance and speed per device
df['distance_meters'] = df.groupby('device_id').apply(lambda x: [
    haversine(x['latitude'].iloc[i], x['longitude'].iloc[i],
              x['latitude'].iloc[i+1], x['longitude'].iloc[i+1])
    for i in range(len(x)-1)
] + [0.0] # Append 0 for the last point, or NaN based on desired behavior
).explode().reset_index(drop=True)

df['time_diff_seconds'] = df.groupby('device_id')['timestamp'].diff().dt.total_seconds()

# For the first entry of each device, time_diff_seconds will be NaN. Set distance_meters to 0.0 for these.
# And speed will be 0 for these. Then fill remaining NaNs with 0.
df.loc[df['time_diff_seconds'].isnull(), 'distance_meters'] = 0.0
df['time_diff_seconds'] = df['time_diff_seconds'].fillna(0)

# Calculate speed (meters per second)
# Avoid division by zero for points where time_diff_seconds is 0
df['speed_mps'] = df.apply(lambda row: row['distance_meters'] / row['time_diff_seconds'] if row['time_diff_seconds'] > 0 else 0,
                           axis=1)

print("DataFrame with calculated distances and speeds:")
print(df[['latitude', 'longitude', 'distance_meters', 'time_diff_seconds', 'speed_mps']].head(10))


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Radius of Earth in meters

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

# Create new columns for lagged latitude and longitude within each device group
df['prev_latitude'] = df.groupby('device_id')['latitude'].shift(1)
df['prev_longitude'] = df.groupby('device_id')['longitude'].shift(1)

# Calculate distance_meters using the haversine function
# Apply haversine where previous data exists (i.e., not the first point for a device), otherwise set to 0.0
df['distance_meters'] = df.apply(
    lambda row: haversine(row['prev_latitude'], row['prev_longitude'], row['latitude'], row['longitude'])
    if pd.notna(row['prev_latitude']) else 0.0,
    axis=1
)

# Calculate time difference in seconds using the DataFrame index (which is 'timestamp')
df['time_diff_seconds'] = df.groupby('device_id').apply(
    lambda x: x.index.to_series().diff().dt.total_seconds()
).reset_index(level=0, drop=True) # Reset level 0 (device_id) to align with df's index
df['time_diff_seconds'] = df['time_diff_seconds'].fillna(0) # Fill NaN for the first point of each device

# Calculate speed (meters per second)
# Avoid division by zero for points where time_diff_seconds is 0
df['speed_mps'] = df.apply(
    lambda row: row['distance_meters'] / row['time_diff_seconds'] if row['time_diff_seconds'] > 0 else 0,
    axis=1
)

# Drop the temporary 'prev_' columns
df = df.drop(columns=['prev_latitude', 'prev_longitude'])

print("DataFrame with calculated distances and speeds:")
print(df[['latitude', 'longitude', 'distance_meters', 'time_diff_seconds', 'speed_mps']].head(10))

In [ ]:
import numpy as np

df['acceleration_magnitude'] = np.sqrt(
    df['acceleration_x']**2 +
    df['acceleration_y']**2 +
    df['acceleration_z']**2
)

print("DataFrame with calculated acceleration magnitude:")
print(df[['acceleration_x', 'acceleration_y', 'acceleration_z', 'acceleration_magnitude']].head())

In [ ]:
window_size = 3 # Represents 3 minutes of data for a minute-frequency dataset

# Apply rolling mean to latitude and longitude, grouped by device_id
df['latitude_smoothed'] = df.groupby('device_id')['latitude'].transform(lambda x: x.rolling(window=window_size, min_periods=1, center=True).mean())
df['longitude_smoothed'] = df.groupby('device_id')['longitude'].transform(lambda x: x.rolling(window=window_size, min_periods=1, center=True).mean())

print(f"DataFrame with smoothed latitude and longitude using a rolling mean with window size = {window_size}:")
print(df[['latitude', 'longitude', 'latitude_smoothed', 'longitude_smoothed']].head(10))

In [ ]:
stop_speed_threshold_mps = 0.5  # meters per second
min_stop_duration_seconds = 180 # 3 minutes

df['mobility_state'] = 'move'

# Group by device_id to process each device's trajectory independently
for device_id, group in df.groupby('device_id'):
    stop_start_index = None

    for i in range(len(group)):
        # Check if the current point is a potential stop
        if group['speed_mps'].iloc[i] < stop_speed_threshold_mps:
            if stop_start_index is None:
                stop_start_index = i # Mark the beginning of a potential stop
        else:
            # If speed exceeds threshold, reset stop_start_index
            if stop_start_index is not None:
                stop_start_index = None

        # If we are in a potential stop segment, check its duration
        if stop_start_index is not None:
            current_time = group.index[i]
            start_time = group.index[stop_start_index]
            duration = (current_time - start_time).total_seconds()

            if duration >= min_stop_duration_seconds:
                # Mark all points from stop_start_index to current_index as 'stop'
                df.loc[(df.index >= start_time) & (df.index <= current_time) & (df['device_id'] == device_id), 'mobility_state'] = 'stop'

print("Value counts for 'mobility_state' column:")
print(df['mobility_state'].value_counts())

print("First 10 rows with 'mobility_state' column:")
print(df[['speed_mps', 'mobility_state']].head(10))

In [ ]:
import numpy as np

# Filter for rows where mobility_state is 'stop'
df_stops = df[df['mobility_state'] == 'stop'].copy()

# Initialize an empty DataFrame for POIs in case no stops are found
pois = pd.DataFrame(columns=['device_id', 'poi_event_id', 'poi_latitude', 'poi_longitude',
                              'start_time', 'end_time', 'visit_duration_seconds', 'number_of_points'])

if not df_stops.empty:
    # Assign a unique ID to each consecutive block of 'stop' within each device.
    # This handles cases where a device stops, moves, then stops again, treating them as distinct POI events.
    df_stops['poi_event_id'] = (df_stops.groupby('device_id')['mobility_state']
                                 .transform(lambda x: (x != x.shift()).cumsum()))

    # Now group by device_id and the newly created poi_event_id to aggregate POI information
    pois = df_stops.groupby(['device_id', 'poi_event_id']).agg(
        poi_latitude=('latitude_smoothed', 'mean'),
        poi_longitude=('longitude_smoothed', 'mean'),
        start_time=('timestamp', 'min'),
        end_time=('timestamp', 'max'),
        number_of_points=('device_id', 'size')
    ).reset_index()

    # Calculate the total duration of each visit to a POI
    pois['visit_duration_seconds'] = (pois['end_time'] - pois['start_time']).dt.total_seconds()

    # Filter out events that do not meet the minimum duration for a stop, as defined previously.
    # This is a safeguard, as the initial stop detection logic should have already considered this.
    pois = pois[pois['visit_duration_seconds'] >= min_stop_duration_seconds].reset_index(drop=True)

    print("Identified Points of Interest (POIs):")
    print(pois)
else:
    print("No 'stop' events were detected in the data after applying the defined thresholds.")
    print("This means no Points of Interest (POIs) could be identified from stop locations.")
    print("An empty POI DataFrame `pois` has been created.")

# The `pois` DataFrame (potentially empty) is now available for further use.


In [ ]:
route_features = df.groupby('device_id').agg(
    start_latitude=('latitude_smoothed', 'first'),
    start_longitude=('longitude_smoothed', 'first'),
    end_latitude=('latitude_smoothed', 'last'),
    end_longitude=('longitude_smoothed', 'last'),
    avg_latitude=('latitude_smoothed', 'mean'),
    avg_longitude=('longitude_smoothed', 'mean')
).reset_index()

print("Extracted route features for clustering:")
print(route_features.head())

# Prepare features for clustering
X = route_features[['start_latitude', 'start_longitude', 'end_latitude', 'end_longitude', 'avg_latitude', 'avg_longitude']].values

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Standardize the features before clustering, as K-Means is sensitive to scale
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply K-Means clustering. Choosing n_clusters = 2 for demonstration.
# In a real scenario, one would use methods like the elbow method or silhouette score to find optimal K.
n_clusters = 2
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10) # n_init to suppress warning
route_features['route_cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means Clustering applied with {n_clusters} clusters.")
print("Route features with cluster assignments:")
print(route_features)

# Optional: Visualize clusters (e.g., using principal components or a subset of features)
# For simplicity, visualizing based on average lat/lon
plt.figure(figsize=(8, 6))
sns.scatterplot(x='avg_longitude', y='avg_latitude', hue='route_cluster', data=route_features, palette='viridis', s=100)
plt.title('Route Clusters based on Average Location')
plt.xlabel('Average Longitude')
plt.ylabel('Average Latitude')
plt.grid(True)
plt.show()

print("Summary of route clusters:")
print(route_features['route_cluster'].value_counts())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Scatter plot of smoothed trajectories
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='longitude_smoothed',
    y='latitude_smoothed',
    hue='device_id',
    data=df,
    palette='viridis',
    s=50,
    alpha=0.7
)
plt.title('Smoothed Trajectories of Each Device')
plt.xlabel('Smoothed Longitude')
plt.ylabel('Smoothed Latitude')
plt.legend(title='Device ID')
plt.grid(True)
plt.show()

# 3. Histogram of speed distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['speed_mps'], kde=True, bins=20)
plt.title('Distribution of Speed')
plt.xlabel('Speed (m/s)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 4. Histogram of acceleration magnitude distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['acceleration_magnitude'], kde=True, bins=20)
plt.title('Distribution of Acceleration Magnitude')
plt.xlabel('Acceleration Magnitude (m/s^2)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 5. Scatter plot for clustered routes
plt.figure(figsize=(12, 10))

# Plot start points
sns.scatterplot(
    x='start_longitude',
    y='start_latitude',
    hue='route_cluster',
    data=route_features,
    palette='tab10',
    marker='o',
    s=150,
    label='Start Point',
    legend='full'
)

# Plot end points
sns.scatterplot(
    x='end_longitude',
    y='end_latitude',
    hue='route_cluster',
    data=route_features,
    palette='tab10',
    marker='X',
    s=150,
    label='End Point',
    legend=False # Do not duplicate legend entries
)

# Draw lines connecting start and end points for each route
for index, row in route_features.iterrows():
    plt.plot(
        [row['start_longitude'], row['end_longitude']],
        [row['start_latitude'], row['end_latitude']],
        color=sns.color_palette('tab10')[row['route_cluster']],
        linestyle='--',
        alpha=0.6
    )

plt.title('Clustered Route Start and End Points')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.legend(title='Route Cluster')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

# 2. Scatter plot of smoothed trajectories
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='longitude_smoothed',
    y='latitude_smoothed',
    hue='device_id',
    data=df,
    palette='viridis',
    s=50,
    alpha=0.7
)
plt.title('Smoothed Trajectories of Each Device')
plt.xlabel('Smoothed Longitude')
plt.ylabel('Smoothed Latitude')
plt.legend(title='Device ID')
plt.grid(True)
plt.show()

# 3. Histogram of speed distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['speed_mps'], kde=True, bins=20)
plt.title('Distribution of Speed')
plt.xlabel('Speed (m/s)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 4. Histogram of acceleration magnitude distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['acceleration_magnitude'], kde=True, bins=20)
plt.title('Distribution of Acceleration Magnitude')
plt.xlabel('Acceleration Magnitude (m/s^2)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 5. Scatter plot for clustered routes
plt.figure(figsize=(12, 10))
ax = plt.gca() # Get current axes

# Plot start points - suppress default legend
sns.scatterplot(
    x='start_longitude',
    y='start_latitude',
    hue='route_cluster',
    data=route_features,
    palette='tab10',
    marker='o',
    s=150,
    ax=ax,
    legend=False
)

# Plot end points - suppress default legend
sns.scatterplot(
    x='end_longitude',
    y='end_latitude',
    hue='route_cluster',
    data=route_features,
    palette='tab10',
    marker='X',
    s=150,
    ax=ax,
    legend=False
)

# Draw lines connecting start and end points for each route
for index, row in route_features.iterrows():
    ax.plot(
        [row['start_longitude'], row['end_longitude']],
        [row['start_latitude'], row['end_latitude']],
        color=sns.color_palette('tab10')[row['route_cluster']],
        linestyle='--',
        alpha=0.6,
        zorder=0
    )

ax.set_title('Clustered Route Start and End Points')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True)

# Manually create legend handles for route_cluster (hue)
unique_clusters = sorted(route_features['route_cluster'].unique())
palette = sns.color_palette('tab10', n_colors=len(unique_clusters))
cluster_handles = [Line2D([0], [0], marker='o', color='w', label=f'Cluster {c}',
                          markerfacecolor=palette[i], markersize=10)
                   for i, c in enumerate(unique_clusters)]

# Manually create legend handles for start/end points (markers)
marker_handles = [
    Line2D([0], [0], marker='o', color='w', label='Start Point',
           markerfacecolor='black', markersize=10),
    Line2D([0], [0], marker='X', color='w', label='End Point',
           markerfacecolor='black', markersize=10)
]

# Combine all handles and labels for a single legend
all_handles = cluster_handles + marker_handles

ax.legend(handles=all_handles, title='Legend')
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# Check if there's more than one cluster to calculate silhouette score
if len(route_features['route_cluster'].unique()) > 1:
    # Calculate the Silhouette Score
    silhouette_avg = silhouette_score(X_scaled, route_features['route_cluster'])
    print(f"The Silhouette Score for the route clustering is: {silhouette_avg:.2f}")
else:
    print("Cannot calculate Silhouette Score: only one cluster found or insufficient data points for distinct clusters.")